# Load a bunch of annoying constants

In [3]:
from pathlib import Path
import polars as pl
import numpy as np
from dataclasses import dataclass
import json
from typing import Dict, Any

CELL_DIVERSITY_BASE_PATH = Path(
    "/rxrx/data/valence/phenomics/cross_cell_line__brightfield__pretrain__v1_0"
)
CELL_DIVERSITY_OBS_PATH = (
    CELL_DIVERSITY_BASE_PATH
    / "cross_cell_line__brightfield__pretrain__v1_0_obs.parquet"
)
INTERNAL_BENCHMARKING_BASE_PATH = Path("/rxrx/data/valence/internal_benchmarking/")

COMPOUND_CONTROLS = {
    "REC-0000058",
    "REC-0000069",
    "REC-0000098",
    "REC-0000219",
    "REC-0000370",
    "REC-0000375",
    "REC-0000391",
    "REC-0000493",
    "REC-0000584",
    "REC-0000590",
    "REC-0000671",
    "REC-0000706",
    "REC-0000737",
    "REC-0000741",
    "REC-0000773",
    "REC-0000842",
    "REC-0000854",
    "REC-0000867",
    "REC-0000889",
    "REC-0000959",
    "REC-0000970",
    "REC-0001029",
    "REC-0001106",
    "REC-0001148",
    "REC-0001183",
    "REC-0001214",
    "REC-0001246",
    "REC-0001249",
    "REC-0001250",
    "REC-0001258",
    "REC-0001289",
    "REC-0001290",
    "REC-0001610",
    "REC-0001619",
    "REC-0001655",
    "REC-0001721",
    "REC-0001730",
    "REC-0001744",
    "REC-0001907",
    "REC-0002007",
    "REC-0002082",
    "REC-0002101",
    "REC-0002178",
    "REC-0003420",
    "REC-0003935",
    "REC-0003980",
    "REC-0003982",
    "REC-0004011",
    "REC-0004273",
    "REC-0004419",
}


VCB_OBS_PATH = (
    INTERNAL_BENCHMARKING_BASE_PATH
    / "vcds1"
    / "drugscreen__cell_paint__v1_2"
    / "drugscreen__cell_paint__v1_2_obs.parquet"
)
TEST_OBS_PATH = (
    INTERNAL_BENCHMARKING_BASE_PATH
    / "vcds1"
    / "prelim_cross_cell_line__brightfield__v1_0"
    / "prelim_cross_cell_line__brightfield__v1_0_obs.parquet"
)

COMMON_COLUMNS = [
    "path",
    "image_type",
    "shape",
    "source",
    "image_type_id",
    "experiment",
    "experiment_id",
    "cell_condition",
    "cell_condition_id",
    "comp_id",
    "c_concs",
]

COMPOUND_PATTERN = r"^REC-\d+$"
TARGET_WELLS = 1000
TRAIN_FRACTION = 0.9
METADATA_SPLIT_SEED = 123456
OBS_SPLIT_SEED = 123456789

class Tokenizer:
    def __init__(self):
        self.token_to_id = {}
        self.id_to_token = []

    def fit(self, df: pl.DataFrame):
        unique_tokens = df.sort()
        for i, token in enumerate(unique_tokens):
            self.token_to_id[token] = i
            self.id_to_token.append(token)
        return self

    def transform(self, x):
        if isinstance(x, str):
            x = [x]
        return [self.token_to_id[token] for token in np.array(x).flatten()]
    
    def __len__(self):
        return len(self.id_to_token)

    def __call__(self, x):
        return self.transform(x)

def assign_random_split(
    df: pl.DataFrame, train_fraction: float, seed: int
) -> pl.DataFrame:
    """Assign a reproducible train/val split."""
    n_train = int(df.height * train_fraction)
    return df.with_columns(
        pl.when(pl.int_range(0, pl.len()).shuffle(seed=seed) < n_train)
        .then(pl.lit("train"))
        .otherwise(pl.lit("val"))
        .alias("split")
    )

# Images with zarrs that are known to be corrupted
SKIP_IMAGES = pl.read_csv(
    "/mnt/ps/home/CORP/jason.hartford/project/big-x/joint-model/metadata/zarr_scan_shards/skip_paths_all.csv"
)["path"].to_list()

@dataclass(frozen=True)
class SplitIndices:
    finetune: np.ndarray
    validation: np.ndarray
    test: np.ndarray

def load_vcb_split_indices(splits_path: Path) -> tuple[Dict[str, Any], SplitIndices]:
    """Load finetune/validation/test split indices."""
    with splits_path.open() as handle:
        splits_json = json.load(handle)
    first_fold = splits_json["folds"][0]
    split_indices = SplitIndices(
        finetune=np.array(first_fold["finetune"]),
        validation=np.array(first_fold["validation"]),
        test=np.array(first_fold["test"]),
    )
    return splits_json, split_indices

# Load the VCB splits
SPLITS_PATH = (
    INTERNAL_BENCHMARKING_BASE_PATH
    / "vcb"
    / "splits"
    / "drugscreen__cell_paint__v1_2"
    / "split_compound_random__v1.json"
)
_, vcb_splits = load_vcb_split_indices(SPLITS_PATH)

In [4]:
df = pl.read_parquet('/rxrx/data/microscopy/photosynthetic_metadata/95M_filtered_6chan_rp006.parquet')
old_shape = df.shape
print(f"Original shape: {old_shape}")
df = df.filter(~pl.col("path").is_null())
new_shape = df.shape
print(f"Dropped {old_shape[0] - new_shape[0]} rows with null paths")
print(f"New shape: {new_shape}")
old_shape = new_shape
df = df.filter(pl.col("shape") == [2048, 2048, 6])
new_shape = df.shape
print(f"Dropped {old_shape[0] - new_shape[0]} rows with shape != [2048, 2048, 6]")
print(f"New shape: {new_shape}")

name1 = "bf_huvec_whole_genome_no_empty_v0.parquet"
name2 = "rptec_metadata_041625_no_96hr_no_empty.parquet"
base_path = "/rxrx/data/microscopy/photosynthetic_metadata/"
df_bf1 = pl.read_parquet(f"{base_path}{name1}")
df_bf2 = pl.read_parquet(f"{base_path}{name2}")
cols = list(set(df_bf1.columns).intersection(set(df_bf2.columns)))
df_bf = pl.concat([df_bf1[cols], df_bf2[cols]])

df_bf = df_bf.with_columns(
    pl.lit(False).alias("is_rp006"),
    pl.lit("brightfield_3channel").alias("image_type"),
    pl.lit([2048, 2048, 3]).alias("shape"),
)
sorted_cols = sorted(df_bf.columns)
df = pl.concat([df_bf[sorted_cols], df[sorted_cols]])
new_shape = df.shape
print(f"New shape: {new_shape}")
print(f"Added {old_shape[0]- df.shape[0]} rows from brightfield metadata")
df = df.with_columns(split=pl.lit("train")) # make everything train for now.

# Match things up with VCB data

df = df.rename({
        "path": "image_path",
        "experiment": "experiment_label",
        "address": "well_address"
    })

df = df.with_columns(
    concentration=pl.col("concentration") * 1000
)

Original shape: (92764542, 20)
Dropped 0 rows with null paths
New shape: (92764542, 20)
Dropped 33472396 rows with shape != [2048, 2048, 6]
New shape: (59292146, 20)
New shape: (75023432, 20)
Added 17741110 rows from brightfield metadata


In [5]:
def get_sampled_ids(df, cell_type, sample_fraction=0.1):
    # 1. Filter by cell type
    # 2. Transform rec_id (sort/join) to ensure consistent string representation
    # 3. Get unique values
    # 4. Sample
    return (
        df.filter(pl.col('cell_type') == cell_type)
          .select(pl.col('rec_id_string'))
          .to_series()
          .unique()
          .sample(fraction=sample_fraction, shuffle=True)
    )

df = df.with_columns(
    rec_id_string = pl.col('rec_id').list.sort().list.join('-')
)

arpe_valid_ids = get_sampled_ids(df, 'ARPE19')
rptec_valid_ids = get_sampled_ids(df, 'RPTEC')
huvec_valid_ids = get_sampled_ids(df, 'HUVEC_TNFalpha')
df = df.with_columns(
    pl.when(pl.col("rec_id_string").is_in(arpe_valid_ids) &  (pl.col("cell_type") == 'ARPE19')).then(pl.lit("valid_arpe19"))
      .when(pl.col("rec_id_string").is_in(rptec_valid_ids) & (pl.col("cell_type") == 'RPTEC')).then(pl.lit("valid_reptec"))
      .when(pl.col("rec_id_string").is_in(huvec_valid_ids) & (pl.col("cell_type") == 'HUVEC_TNFalpha')).then(pl.lit("valid_huvec_tnfalpha"))
      .otherwise(pl.col("split")) # or pl.lit("train")
      .alias("split")
)

/tmp/ipykernel_62165/1802712642.py:21: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df = df.with_columns(


In [6]:
df = df.with_row_index("row_idx") 
bf3_indices = df.filter(pl.col("image_type") == "brightfield_3channel").select("row_idx")
sampled_bf3_indices = bf3_indices.sample(fraction=0.002, shuffle=True) # just saving approximately 32 000 rows for FID

cp_indices = df.filter(pl.col("image_type") == "cellpaint").select("row_idx")
sampled_cp_indices = cp_indices.sample(fraction=0.0005, shuffle=True) # just saving approximately 32 000 rows for FID

df = df.with_columns(
    pl.when(pl.col("row_idx").is_in(sampled_bf3_indices["row_idx"]))
      .then(pl.lit("valid_bf3_iid"))
      .when(pl.col("row_idx").is_in(sampled_cp_indices["row_idx"]))
      .then(pl.lit("valid_cp_iid"))
      .otherwise(pl.col("split"))
      .alias("split")
).drop("row_idx") # Clean up the temporary index
df['split'].value_counts()

/tmp/ipykernel_62165/2351498326.py:8: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df = df.with_columns(


split,count
str,u32
"""valid_reptec""",742344
"""valid_bf3_iid""",31462
"""valid_huvec_tnfalpha""",85260
"""train""",73205604
"""valid_arpe19""",929831
"""valid_cp_iid""",28931


In [7]:
cell_type_diversity_df_train = (pl.read_parquet(str(CELL_DIVERSITY_OBS_PATH)).with_columns(
        pl.lit("brightfield_3channel").alias("image_type"),
        pl.lit("celltype_diversity").alias("source"),
        pl.lit([2048, 2048, 3]).alias("shape"),
        pl.lit("train").alias("split"),
        rec_id=pl.col("perturbations").list.eval(
            pl.element().struct.field("source_id")
        ),
        concentration=pl.col("perturbations").list.eval(
            pl.element().struct.field("concentration").cast(pl.Float64).cast(pl.String)
        ),
    )
)
print("Cell type diversity data (aka Charles' data)")
rec_id_tokenizer = Tokenizer().fit(cell_type_diversity_df_train['rec_id'].explode().unique())
concentration_tokenizer = Tokenizer().fit(cell_type_diversity_df_train['concentration'].explode().unique())
cell_type_tokenizer = Tokenizer().fit(cell_type_diversity_df_train['cell_type'].unique())
image_type_tokenizer = Tokenizer().fit(cell_type_diversity_df_train['image_type'].unique())
experiment_tokenizer = Tokenizer().fit(cell_type_diversity_df_train['experiment_label'].unique())

print("Unique rec_ids: ", len(rec_id_tokenizer))
print("Unique concentrations: ", len(concentration_tokenizer))
print("Unique cell_types: ", len(cell_type_tokenizer))
print("Unique image_types: ", len(image_type_tokenizer))
print("Unique experiments: ", len(experiment_tokenizer))

Cell type diversity data (aka Charles' data)
Unique rec_ids:  6655
Unique concentrations:  5
Unique cell_types:  17
Unique image_types:  1
Unique experiments:  158


### First annoying code

Preparing the "virtual plate" of one plate of data from the new cell type

In [8]:
target_wells = 1500

cell_type_diversity_df_test = (pl.read_parquet(str(TEST_OBS_PATH)).with_columns(
        pl.lit("brightfield_3channel").alias("image_type"),
        pl.lit("celltype_diversity").alias("source"),
        pl.lit([2048, 2048, 3]).alias("shape"),
        pl.lit("test").alias("split"),
        rec_id=pl.col("perturbations").list.eval(
            pl.element().struct.field("source_id")
        ),
        concentration=pl.col("perturbations").list.eval(
            pl.element().struct.field("concentration").cast(pl.Float64).cast(pl.String)
        ),
    )
).with_row_index("row_id")


positive_controls = cell_type_diversity_df_test.filter(
    pl.col("rec_id").list.eval(pl.element().is_in(COMPOUND_CONTROLS)).list.any()
)
empties = cell_type_diversity_df_test.filter(
    pl.col("rec_id").list.eval(pl.element().is_not_null()).list.any().not_()
)
control_counts = positive_controls.group_by("cell_type").len(name="count")

plate_rankings = (
    positive_controls.group_by(
        ["cell_type", "experiment_label", "experiment_plate_number"]
    )
    .len(name="pos_control_density")
    .sort(["cell_type", "pos_control_density"], descending=True)
)

ranked_empties = (
    empties.join(
        plate_rankings,
        on=["cell_type", "experiment_label", "experiment_plate_number"],
        how="left",
    )
    .with_columns(pl.col("pos_control_density").fill_null(0))
    .sort(["cell_type", "pos_control_density"], descending=True)
)

final_parts: list[pl.DataFrame] = []
if positive_controls.height > 0:
    final_parts.append(positive_controls)

for cell_type_data in control_counts.rows(named=True):
    n_needed = max(0, target_wells - cell_type_data["count"])
    if n_needed == 0:
        continue
    empties_subset = (
        ranked_empties.filter(pl.col("cell_type") == cell_type_data["cell_type"])
        .head(n_needed)
        .drop("pos_control_density")
    )
    if empties_subset.height > 0:
        final_parts.append(empties_subset)

if not final_parts:
    final_parts.append(
        ranked_empties.head(target_wells).drop("pos_control_density")
    )

virtual_plate_ids = pl.concat(final_parts)["row_id"]

# Assign split: 'train' for virtual plate, 'test' for the rest
prepared = cell_type_diversity_df_test.with_columns(
    split=pl.when(pl.col("row_id").is_in(virtual_plate_ids))
    .then(pl.lit("train"))
    .otherwise(pl.lit("test"))
)
print("Virtual Test Plate Split:", prepared["split"].value_counts())

cell_type_diversity_df = pl.concat([cell_type_diversity_df_train, prepared[cell_type_diversity_df_train.columns]])
print("Cell type diversity data")
cell_type_diversity_df['split'].value_counts()

Virtual Test Plate Split: shape: (2, 2)
┌───────┬────────┐
│ split ┆ count  │
│ ---   ┆ ---    │
│ str   ┆ u32    │
╞═══════╪════════╡
│ test  ┆ 129432 │
│ train ┆ 3000   │
└───────┴────────┘
Cell type diversity data


/tmp/ipykernel_62165/240643703.py:68: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  prepared = cell_type_diversity_df_test.with_columns(


split,count
str,u32
"""test""",129432
"""train""",865088


# VCB

This is almost exactly the same as the steps above except we use the splits to define train / test / valid.

**Only potential gotcha:** see how I'm adding an id column and check its okay

In [9]:
vcb_obs = pl.read_parquet(str(VCB_OBS_PATH)).with_columns(
    id=pl.arange(0, pl.len()), # NB: I'd appreciate a second set of eyes on this. It assumes that the rows in the splits correspond to the rows in the VCB obs data.
).with_columns(
        pl.lit("cellpaint").alias("image_type"),
        pl.lit("vcb").alias("source"),
        pl.lit([2048, 2048, 6]).alias("shape"),
        split = pl.when(pl.col("id").is_in(vcb_splits.finetune))
        .then(pl.lit("train"))
            .when(pl.col("id").is_in(vcb_splits.validation))
        .then(pl.lit("val"))
        .when(pl.col("id").is_in(vcb_splits.test))
        .then(pl.lit("test"))
        .otherwise(pl.lit(None)),
        rec_id=pl.col("perturbations").list.eval(
            pl.element().struct.field("source_id")
        ),
        concentration=pl.col("perturbations").list.eval(
            pl.element().struct.field("concentration").cast(pl.Float64).cast(pl.String)
        ),
    )

print("VCB data")
rec_id_tokenizer = Tokenizer().fit(vcb_obs['rec_id'].explode().unique())
concentration_tokenizer = Tokenizer().fit(vcb_obs['concentration'].explode().unique())
cell_type_tokenizer = Tokenizer().fit(vcb_obs['cell_type'].unique())
image_type_tokenizer = Tokenizer().fit(vcb_obs['image_type'].unique())
experiment_tokenizer = Tokenizer().fit(vcb_obs['experiment_label'].unique())

print("Unique rec_ids: ", len(rec_id_tokenizer))
print("Unique concentrations: ", len(concentration_tokenizer))
print("Unique cell_types: ", len(cell_type_tokenizer))
print("Unique image_types: ", len(image_type_tokenizer))
print("Unique experiments: ", len(experiment_tokenizer))


VCB data
Unique rec_ids:  2712
Unique concentrations:  22
Unique cell_types:  2
Unique image_types:  1
Unique experiments:  56


# Please Check!

This part is the only really annoying bit - we want to add "one plate of data" to the training set to enable generalization to new perturbations.

I'll be honest - I'm a little unsure of how much is reasonable here because in practice if this worked well, we'd do a whole geneome knockout with some modest set of compounds on each gene knockout - so unlike the cell type diversity task, I don't think you'd do 1 plate per gene (that's 20000 plates!)

In [10]:
# Masks for finding controls
is_neg_or_base = (pl.col("is_negative_control")) | (pl.col("is_base_state"))
is_comp_control = (
    pl.col("rec_id")
    .list.eval(pl.element().is_in(COMPOUND_CONTROLS))
    .list.any()
)

any_control_mask = is_neg_or_base | is_comp_control

vcb_obs = vcb_obs.with_columns(no_comp_rec_id=pl.col("perturbations")
        .list.eval(pl.element().struct.field("source_id"))
        .list.eval(
            pl.element().filter(pl.element().str.contains(COMPOUND_PATTERN).not_())
        )).with_columns(
    cell_condition=pl.format(
        "{}_{}",
        pl.col("cell_type"),
        pl.col("no_comp_rec_id").list.sort().list.join("-"),
    )
)

# 1. Batch Matching (Existing Training Data)
    # Find (Experiment, CellType) pairs in existing train set
train_context = (
    vcb_obs.filter(pl.col("split") == "train")
    .select(["experiment_label", "cell_type"])
    .unique()
)
# Find controls in those same batches
train_matched_controls = (
    vcb_obs.join(train_context, on=["experiment_label", "cell_type"], how="inner")
    .filter(is_neg_or_base)
    .select("id")
)

# 2. Virtual Plate (Val/Test Data)
# Find cell_conditions in val/test
val_test_conditions = (
    vcb_obs.filter(pl.col("split").is_in(["val", "test"]))
    .select("cell_condition")
    .unique()
)
# Filter candidates (Base + Neg + Positive Controls) and match condition
virtual_plate_candidates = vcb_obs.filter(any_control_mask).join(
    val_test_conditions, on="cell_condition", how="inner"
)
# Rank plates by density
plate_rankings = (
    virtual_plate_candidates.group_by(
        ["cell_condition", "experiment_label", "experiment_plate_number"]
    )
    .len(name="density")
    .sort(["cell_condition", "density"], descending=True)
)
# Pick best plate per condition
best_plates = plate_rankings.group_by("cell_condition", maintain_order=True).head(1)
# Get IDs
virtual_plate_controls = virtual_plate_candidates.join(
    best_plates,
    on=["cell_condition", "experiment_label", "experiment_plate_number"],
    how="inner",
).select("id")

print(f"Virtual plate controls adds {virtual_plate_controls.shape[0]} rows to the training set")

# 3. Update Split Column
ids_to_add_to_train = train_matched_controls.vstack(virtual_plate_controls).unique()

vcb_obs = vcb_obs.with_columns(
    split=pl.when(pl.col("id").is_in(ids_to_add_to_train["id"]))
    .then(pl.lit("train"))
    .otherwise(pl.col("split"))
)

Virtual plate controls adds 13762 rows to the training set


/tmp/ipykernel_62165/2331328875.py:70: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  vcb_obs = vcb_obs.with_columns(


# Active Learning Data

In [56]:
import zarr
al = pl.read_parquet("/mnt/ps/home/CORP/jason.hartford/pythondev/photosynthetic/notebooks/AL_metadata.parquet")
al = al.with_columns(
    pl.lit("train").alias("split"),
    n_perts = pl.col("rec_id").list.len()
)
al#.filter(pl.col("n_perts") > 1).filter(pl.col("order_id") == 5947)


batch_id,experiment,plate,order_id,read_id,address,site,gene,rec_id,control,concentration,perturbation_type,cell_type,perturbation,split,path,__index_level_0__,n_perts
i64,str,i64,i64,i64,str,i64,list[str],list[str],bool,list[f64],list[str],str,str,str,str,i64,u32
5356,"""AL-051-HUVEC-a""",1,5947,283,"""AA02""",1,"[""FBXO5""]","[""REC-GRNA-0009362""]",false,[0.1],"[""GUIDE""]","""HUVEC""","""rec_id=REC-GRNA-0009362;concen…","""train""","""/rxrx/data/microscopy/zarr_512…",0,1
5356,"""AL-051-HUVEC-a""",1,5947,283,"""AA03""",1,"["""", ""LAMTOR2""]","[""REC-0083196"", ""REC-GRNA-0146248""]",false,"[10.0, 0.1]","[""COMPOUND"", ""GUIDE""]","""HUVEC""","""rec_id=REC-0083196,REC-GRNA-01…","""train""","""/rxrx/data/microscopy/zarr_512…",1,2
5356,"""AL-051-HUVEC-a""",1,5947,283,"""AA04""",1,"[""TRIM22""]","[""REC-GRNA-0010006""]",false,[0.1],"[""GUIDE""]","""HUVEC""","""rec_id=REC-GRNA-0010006;concen…","""train""","""/rxrx/data/microscopy/zarr_512…",2,1
5356,"""AL-051-HUVEC-a""",1,5947,283,"""AA05""",1,"[""MCL1""]","[""REC-GRNA-0015156""]",false,[0.1],"[""GUIDE""]","""HUVEC""","""rec_id=REC-GRNA-0015156;concen…","""train""","""/rxrx/data/microscopy/zarr_512…",3,1
5356,"""AL-051-HUVEC-a""",1,5947,283,"""AA06""",1,"["""", """"]","[""REC-0064071"", ""REC-1156464""]",false,"[0.1, 10.0]","[""COMPOUND"", ""COMPOUND""]","""HUVEC""","""rec_id=REC-0064071,REC-1156464…","""train""","""/rxrx/data/microscopy/zarr_512…",4,2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
9535,"""AL001-pheno-multipert_p36-H-a""",12,8551,586,"""K02""",1,"[""GZMH""]","[""REC-GRNA-0139827""]",false,[0.1],"[""GUIDE""]","""HUVEC""","""rec_id=REC-GRNA-0139827;concen…","""train""","""/rxrx/data/microscopy/zarr_512…",1695450,1
9535,"""AL001-pheno-multipert_p36-H-a""",12,8551,586,"""K03""",1,[],[],true,[],[],"""HUVEC""","""rec_id=;concentration=""","""train""","""/rxrx/data/microscopy/zarr_512…",1695451,0
9535,"""AL001-pheno-multipert_p36-H-a""",12,8551,586,"""K04""",1,[],[],true,[],[],"""HUVEC""","""rec_id=;concentration=""","""train""","""/rxrx/data/microscopy/zarr_512…",1695452,0


# Now lets merge it all together

In [11]:
df = df.with_columns(
    pl.concat_str(['plate', 'order_id' ,'read_id'], separator='_').alias('plate_order_read_id'),
    concentration=pl.col('concentration').list.eval(
    pl.element().cast(pl.String),
))
cols = sorted(list(set(df.columns).intersection(set(vcb_obs.columns))))
combined = pl.concat([df[cols], vcb_obs[cols], cell_type_diversity_df_train[cols]])
combined = combined.filter(~pl.col("image_path").is_in(SKIP_IMAGES)) # skip images that are known to be corrupted

In [12]:
combined.with_columns(pl.concat_str(['plate_order_read_id', 'well_address'], separator=':').alias('obs_id')).write_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/joint-model/metadata/pretraining_v3.parquet')

In [13]:
print("Combined data")
rec_id_tokenizer = Tokenizer().fit(combined['rec_id'].explode().unique())
concentration_tokenizer = Tokenizer().fit(combined['concentration'].explode().unique())
cell_type_tokenizer = Tokenizer().fit(combined['cell_type'].unique())
image_type_tokenizer = Tokenizer().fit(combined['image_type'].unique())
experiment_tokenizer = Tokenizer().fit(combined['experiment_label'].unique())
well_tokenizer = Tokenizer().fit(combined['well_address'].unique())

print("Unique rec_ids: ", len(rec_id_tokenizer))
print("Unique concentrations: ", len(concentration_tokenizer))
print("Unique cell_types: ", len(cell_type_tokenizer))
print("Unique image_types: ", len(image_type_tokenizer))
print("Unique experiments: ", len(experiment_tokenizer))
print("Unique well_addresses: ", len(well_tokenizer))


Combined data


Unique rec_ids:  613486
Unique concentrations:  329
Unique cell_types:  43
Unique image_types:  4
Unique experiments:  8688
Unique well_addresses:  1531


In [1]:
import polars as pl
combined = pl.read_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/joint-model/metadata/pretraining_v2.parquet')

In [40]:
combined

cell_type,concentration,experiment_label,image_path,image_type,order_id,plate_order_read_id,read_id,rec_id,shape,split,well_address
str,list[str],str,str,str,i64,str,i64,list[str],list[i64],str,str
"""HUVEC""","[""100.0""]","""pheno-WholeGenome_p1-H-a""","""/rxrx/data/microscopy/zarr_512…","""brightfield_3channel""",6429,"""1_6429_294""",294,"[""REC-GRNA-0118831""]","[2048, 2048, 3]","""train""","""AA02"""
"""HUVEC""","[""100.0""]","""pheno-WholeGenome_p1-H-a""","""/rxrx/data/microscopy/zarr_512…","""brightfield_3channel""",6429,"""1_6429_294""",294,"[""REC-GRNA-0119199""]","[2048, 2048, 3]","""train""","""AA03"""
"""HUVEC""","[""100.0""]","""pheno-WholeGenome_p1-H-a""","""/rxrx/data/microscopy/zarr_512…","""brightfield_3channel""",6429,"""1_6429_294""",294,"[""REC-GRNA-0007820""]","[2048, 2048, 3]","""train""","""AA04"""
"""HUVEC""","[""100.0""]","""pheno-WholeGenome_p1-H-a""","""/rxrx/data/microscopy/zarr_512…","""brightfield_3channel""",6429,"""1_6429_294""",294,"[""REC-GRNA-0118019""]","[2048, 2048, 3]","""train""","""AA05"""
"""HUVEC""","[""100.0""]","""pheno-WholeGenome_p1-H-a""","""/rxrx/data/microscopy/zarr_512…","""brightfield_3channel""",6429,"""1_6429_294""",294,"[""REC-GRNA-0117128""]","[2048, 2048, 3]","""train""","""AA06"""
…,…,…,…,…,…,…,…,…,…,…,…
"""HACAT""","[""25.0""]","""Hooke1-Nointron-KCE_p3-HACAT-d""","""/rxrx/data/microscopy/zarr_512…","""brightfield_3channel""",11849,"""486710_11849_9965""",9965,"[""REC-0003648""]","[2048, 2048, 3]","""train""","""Z43"""
"""HACAT""","[""25.0""]","""Hooke1-Nointron-KCE_p3-HACAT-d""","""/rxrx/data/microscopy/zarr_512…","""brightfield_3channel""",11849,"""486710_11849_9965""",9965,"[""REC-0003610""]","[2048, 2048, 3]","""train""","""Z44"""
"""HACAT""","[""25.0""]","""Hooke1-Nointron-KCE_p3-HACAT-d""","""/rxrx/data/microscopy/zarr_512…","""brightfield_3channel""",11849,"""486710_11849_9965""",9965,"[""REC-0000075""]","[2048, 2048, 3]","""train""","""Z45"""


In [3]:
import polars as pl
pl.read_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/metrics/big-img-dart-testing/step_250000/dart/features/dart/obs.parquet')

zarr_index_generated_raw_counts,plate_order_read_id,read_id,barcode,cumulative_transfer_ul,working_stock_protocol_id,perturbations,experiment_label,well_address,plate_id,staining_protocol_id,order_id,reference_time,cell_type,bioassay_end_post_reference,experiment_plate_number,experiment_protocol_id,obs_id,sample_id,batch_split,batch_center,is_negative_control,image_path,is_base_state,drugscreen_query,plate_disease_model,plate_disease_model_symbol,id,image_type,source,shape,split,rec_id,concentration_dart
u32,str,i64,str,"decimal[9,9]",i64,list[struct[26]],str,str,i64,i64,i64,date,str,i64,i64,i64,str,str,str,i64,bool,str,bool,bool,str,str,i64,str,str,list[i64],str,list[str],list[str]
0,"""101361_-1_-1""",-1,"""p79007""",0.010000000,6,"[{null,null,null,null,null,50000.000000000,""nM"",""KO"",""ENSG00000134086"",null,""VHL"",""110fd7aaadf1302ac83c242b3dc84d8f"",null,""/Alt1/rCrGrCrUrCrUrUrUrCrArGrArGrUrArUrArCrArCrGrUrUrUrUrArGrArGrCrUrArUrGrCrU/Alt2/"",0,null,null,null,""REC-GRNA-0007122"",null,null,""CRISPR-Cas9"",""CGCTCTTTCAGAGTATACAC"",""exon"",""genetic"",""query""}, {null,null,null,"""",null,10000.000000000,""nM"",null,null,null,null,null,null,null,72,""RPXVIAFEQBNEAX-UHFFFAOYSA-N"",""232.155"",""N#Cc1cc2[nH]c(=O)c(=O)[nH]c2cc1[N+](=O)[O-]"",""REC-0004974"",""RPXVIAFEQBNEAX-UHFFFAOYSA-N"",""[O-][N+](=O)C1=C(C#N)C=C2NC(=O)C(=O)NC2=C1"",null,null,null,""compound"",""query""}]","""VHL-Core01-H-C700-a""","""AA02""",101361,67,-1,2019-10-28,"""HUVEC""",null,8,1378,"""101361_-1_-1:AA02""","""101361:AA02""","""VHL-Core01-H-C700-a""",101361,false,"""/rxrx/data/microscopy/zarr_512…",false,true,"""ENSG00000134086""","""VHL""",0,"""cellpaint""","""vcb""","[2048, 2048, 6]","""train""","[""REC-GRNA-0007122"", ""REC-0004974""]","[""50000.0"", ""10000.0""]"
1,"""101361_-1_-1""",-1,"""p79007""",0.010000000,6,"[{null,null,null,null,null,50000.000000000,""nM"",""KO"",""ENSG00000134086"",null,""VHL"",""110fd7aaadf1302ac83c242b3dc84d8f"",null,""/Alt1/rCrGrCrUrCrUrUrUrCrArGrArGrUrArUrArCrArCrGrUrUrUrUrArGrArGrCrUrArUrGrCrU/Alt2/"",0,null,null,null,""REC-GRNA-0007122"",null,null,""CRISPR-Cas9"",""CGCTCTTTCAGAGTATACAC"",""exon"",""genetic"",""query""}, {null,null,null,""119431-25-3"",null,10000.000000000,""nM"",null,null,null,null,null,null,null,72,""GGUSQTSTQSHJAH-UHFFFAOYSA-N"",""347.861"",""OC(CN1CCC(Cc2ccc(F)cc2)CC1)c1ccc(Cl)cc1"",""REC-0006806"",""GGUSQTSTQSHJAH-UHFFFAOYSA-N"",""OC(CN1CCC(CC2=CC=C(F)C=C2)CC1)C1=CC=C(Cl)C=C1"",null,null,null,""compound"",""query""}]","""VHL-Core01-H-C700-a""","""AA03""",101361,67,-1,2019-10-28,"""HUVEC""",null,8,1378,"""101361_-1_-1:AA03""","""101361:AA03""","""VHL-Core01-H-C700-a""",101361,false,"""/rxrx/data/microscopy/zarr_512…",false,true,"""ENSG00000134086""","""VHL""",1,"""cellpaint""","""vcb""","[2048, 2048, 6]","""train""","[""REC-GRNA-0007122"", ""REC-0006806""]","[""50000.0"", ""10000.0""]"
2,"""101361_-1_-1""",-1,"""p79007""",0.010000000,6,"[{null,null,null,null,null,50000.000000000,""nM"",""KO"",""ENSG00000134086"",null,""VHL"",""110fd7aaadf1302ac83c242b3dc84d8f"",null,""/Alt1/rCrGrCrUrCrUrUrUrCrArGrArGrUrArUrArCrArCrGrUrUrUrUrArGrArGrCrUrArUrGrCrU/Alt2/"",0,null,null,null,""REC-GRNA-0007122"",null,null,""CRISPR-Cas9"",""CGCTCTTTCAGAGTATACAC"",""exon"",""genetic"",""query""}, {null,null,null,"""",null,10000.000000000,""nM"",null,null,null,null,null,null,null,72,""JBALRFFXKQPVLT-UHFFFAOYSA-N"",""486.6080000000004"",""COCCOCOc1cc2ccc(-c3ccc(C(=O)O)cc3)cc2cc1C12CC3CC(CC(C3)C1)C2"",""REC-0063019"",""JBALRFFXKQPVLT-UHFFFAOYSA-N"",""COCCOCOC1=CC2=C(C=C(C3=CC=C(C(O)=O)C=C3)C=C2)C=C1C12CC3CC(CC(C3)C1)C2"",null,null,null,""compound"",""query""}]","""VHL-Core01-H-C700-a""","""AA04""",101361,67,-1,2019-10-28,"""HUVEC""",null,8,1378,"""101361_-1_-1:AA04""","""101361:AA04""","""VHL-Core01-H-C700-a""",101361,false,"""/rxrx/data/microscopy/zarr_512…",false,true,"""ENSG00000134086""","""VHL""",2,"""cellpaint""","""vcb""","[2048, 2048, 6]","""train""","[""REC-GRNA-0007122"", ""REC-0063019""]","[""50000.0"", ""1000

In [ ]:
cell_type_diversity_overfit = (pl.read_parquet(str(TEST_OBS_PATH)).with_columns(
        pl.lit("brightfield_3channel").alias("image_type"),
        pl.lit("celltype_diversity").alias("source"),
        pl.lit([2048, 2048, 3]).alias("shape"),
        pl.lit("train").alias("split"),
        pl.concat_str(['plate_order_read_id', 'well_address'], separator=':').alias('obs_id'),
        rec_id=pl.col("perturbations").list.eval(
            pl.element().struct.field("source_id")
        ),
        concentration=pl.col("perturbations").list.eval(
            pl.element().struct.field("concentration").cast(pl.Float64).cast(pl.String)
        ),
    )
).with_row_index("row_id")

dart_overfit = (pl.read_parquet('/rxrx/data/valence/internal_benchmarking/context_vcds1/dart_example__v1_0/dart_example__v1_0_obs.parquet')
.with_columns(
    id=pl.arange(0, pl.len()), # NB: I'd appreciate a second set of eyes on this. It assumes that the rows in the splits correspond to the rows in the VCB obs data.
).with_columns(
        pl.lit("cellpaint").alias("image_type"),
        pl.lit("vcb").alias("source"),
        pl.lit([2048, 2048, 6]).alias("shape"),
        pl.lit("train").alias("split"),
        pl.concat_str(['plate_order_read_id', 'well_address'], separator=':').alias('obs_id'),
        rec_id=pl.col("perturbations").list.eval(
            pl.element().struct.field("source_id")
        ),
        concentration=pl.col("perturbations").list.eval(
            pl.element().struct.field("concentration").cast(pl.Float64).cast(pl.String)
        ),
    )
)
overfit = pl.concat([dart_overfit[combined.columns], cell_type_diversity_overfit[combined.columns]])
overfit.head()

print("Overfitting data")
rec_id_tokenizer = Tokenizer().fit(overfit['rec_id'].explode().unique())
concentration_tokenizer = Tokenizer().fit(overfit['concentration'].explode().unique())
cell_type_tokenizer = Tokenizer().fit(overfit['cell_type'].unique())
image_type_tokenizer = Tokenizer().fit(overfit['image_type'].unique())
experiment_tokenizer = Tokenizer().fit(overfit['experiment_label'].unique())
well_tokenizer = Tokenizer().fit(overfit['well_address'].unique())

print("Unique rec_ids: ", len(rec_id_tokenizer))
print("Unique concentrations: ", len(concentration_tokenizer))
print("Unique cell_types: ", len(cell_type_tokenizer))
print("Unique image_types: ", len(image_type_tokenizer))
print("Unique experiments: ", len(experiment_tokenizer))
print("Unique well_addresses: ", len(well_tokenizer))
overfit.write_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/joint-model/metadata/overfit_data.parquet')

Overfitting data
Unique rec_ids:  8838
Unique concentrations:  9
Unique cell_types:  3
Unique image_types:  2
Unique experiments:  25
Unique well_addresses:  1380


In [10]:
overfit.columns

['cell_type',
 'concentration',
 'experiment_label',
 'image_path',
 'image_type',
 'order_id',
 'plate_order_read_id',
 'read_id',
 'rec_id',
 'shape',
 'split',
 'well_address',
 'obs_id']

In [ ]:
['path', 'cell_type', 'experiment_label', 'image_type', 'well_address', 'rec_id', 'concentration']